### Homework 5: Question search engine

Remeber week01 where you used GloVe embeddings to find related questions? That was.. cute, but far from state of the art. It's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [1]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.5 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.18.2-py3-none-any.whl size=1763310 sha256=55209d6e06ffa47438f0fbfdb9e381ccff37c136a071aa194a33d1be81522b15
  Stored in directory: /root/.cache/pip/wheels/69/ad/2e/e03d4739ddc0417efd8a120c2b9e784005aa226037e558c163
Successfully built deepspeed
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4

### Load data and model

In [2]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

### Tokenize the data

In [4]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [5]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Task 1: evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [6]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [7]:
for batch in val_loader:
     break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nPrediction (probs):', torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
model = model.to(device)

cuda


In [9]:
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=2
)

true_values = []
pred_values = []
for batch in tqdm(val_loader): ####
     # here be your training code
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
      predicted = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        token_type_ids=batch['token_type_ids'])

    pred = torch.argmax(predicted.logits, dim=1).data

    true_values += list(batch['labels'].data.cpu().numpy())
    pred_values += list(pred.cpu().numpy())
#print("Sample batch:", batch)




accuracy = accuracy_score(true_values, pred_values)

  0%|          | 0/2527 [00:00<?, ?it/s]

In [10]:
assert 0.9 < accuracy < 0.91

### Task 2: train the model (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [11]:
model_name = "microsoft/deberta-v3-base"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
model = model.to(device)

cuda


In [13]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [14]:
val_set = qqp_preprocessed['validation']
train_set = qqp_preprocessed['train']

val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=2)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=32, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=2)


In [15]:
from sklearn.metrics import accuracy_score

def train_epoch(train_loader, model, loss_fn, optimizer):
  losses = []
  t = tqdm(train_loader)
  model.train()
  for i, batch in enumerate(t):
      batch_on_dev = {k: v.to(device) for k, v in batch.items()}
      predicted = model(
          input_ids=batch_on_dev['input_ids'],
          attention_mask=batch_on_dev['attention_mask'],
          token_type_ids=batch_on_dev['token_type_ids']
      )

      loss = loss_fn(predicted.logits, batch_on_dev['labels'])
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      if i % 500 == 0:
          t.set_description(f'Curr loss: {loss.item()}')
          t.update()
      losses.append(loss.item())
  return losses


def val_epoch(val_loader, model):
    true_values = []
    pred_values = []
    for batch in tqdm(val_loader):
        batch_on_dev = {k: v.to(device) for k, v in batch.items()}
        predicted = model(
            input_ids=batch_on_dev['input_ids'],
            attention_mask=batch_on_dev['attention_mask'],
            token_type_ids=batch_on_dev['token_type_ids']
        )
        pred = torch.argmax(predicted.logits, dim=-1).data

        pred_values += list(pred.cpu().numpy())
        true_values += list(batch_on_dev['labels'].data.cpu().numpy())

    return accuracy_score(true_values, pred_values)

In [16]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
loss_fn = nn.CrossEntropyLoss()

In [17]:
import numpy as np

NUM_EPOCHS = 1

losses = []
acc = []

for epoch in range(NUM_EPOCHS):
    epoch_losses = train_epoch(train_loader, model, loss_fn, optimizer)
    losses += epoch_losses

    epoch_acc = val_epoch(val_loader, model)
    acc += [epoch_acc]
    print(f'Эпоха {epoch}, итоговый средний loss: {np.mean(epoch_losses)}, accuracy: {epoch_acc}')

  0%|          | 0/11371 [00:00<?, ?it/s]

  0%|          | 0/2527 [00:00<?, ?it/s]

Эпоха 0, итоговый средний loss: 0.25515303094401626, accuracy: 0.9042047984170171


### Task 3: try the full pipeline (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

In [53]:
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_questions(query_question, model, tokenizer, qqp, top_k=5):
    # Переводим модель в режим оценки
    model.eval()

    query_inputs = tokenizer(query_question, return_tensors="pt").to(model.device)
    query_outputs = model(**query_inputs)
    query_logits = query_outputs.logits

    dataset_logits = []
    question_texts = []

    for i, example in enumerate(tqdm(qqp['train'])):
        # Токенизируем первый вопрос из пары
        inputs = tokenizer(
            example['text1'],
            padding='max_length',
            max_length=128,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits.cpu().numpy().flatten()

        dataset_logits.append(logits)
        question_texts.append((example['text1'], example['text2']))

        # Ограничиваем для скорости (можно убрать или изменить)
        if i >= 1000:
            break

    dataset_logits

    # Преобразуем в numpy массивы
    dataset_logits = np.array(dataset_logits)
    query_logits = query_logits.detach().cpu().numpy()
    # Вычисляем косинусное сходство между логитами
    similarities = cosine_similarity(query_logits, dataset_logits)[0]

    # Получаем индексы top_k самых похожих вопросов
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    # Формируем результат
    results = []
    for idx in top_indices:
        similar_question = question_texts[idx][0]
        second_question = question_texts[idx][1]
        similarity_score = similarities[idx]
        results.append((similar_question, similarity_score))

    return results

In [54]:
similar_questions = find_similar_questions(
     "How to learn programming?",
     model,
     tokenizer,
     qqp)

  0%|          | 0/363846 [00:00<?, ?it/s]

In [55]:
similar_questions

[('How is the life of a math student? Could you describe your own experiences?',
  np.float32(1.0000001),
  'Which level of prepration is enough for the exam jlpt5?'),
 ('Is it worth pursuing a graduate degree in medicine?',
  np.float32(1.0000001),
  'Is it worth pursuing a graduate degree in journalism?'),
 ('What makes you a human?',
  np.float32(1.0000001),
  'What makes you truly human?'),
 ('Travel Startups and Companies: What is the difference between TripCase, TripIt and WorldMate?',
  np.float32(1.0000001),
  'Is TripIt worth using?'),
 ('How do I reset my gmail password when they are not highlighting my recovery email option?',
  np.float32(1.0000001),
  'I lost my password with my Gmail account. How do I reset it without the account recovery info?')]

In [68]:
def preprocess_function2(examples, query):
    result = [tokenizer(query, examples['text1'], padding='max_length', max_length=MAX_LENGTH, truncation=True),
              tokenizer(query, examples['text2'], padding='max_length', max_length=MAX_LENGTH, truncation=True)]
    return result

qqp_preprocessed = qqp.map(lambda x: preprocess_function2(examples = x, query="How to learn programming?"), batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

TypeError: TextInputSequence must be str

In [ ]:
qqp_preprocessed

In [62]:
question = "How to learn programming?"

val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=2
)

true_values = []
pred_values = []
for example in tqdm(val_loader): ####
    print(batch)
    break
     # here be your training code
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
      predicted = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        token_type_ids=batch['token_type_ids'])

    pred = torch.argmax(predicted.logits, dim=1).data

    true_values += list(batch['labels'].data.cpu().numpy())
    pred_values += list(pred.cpu().numpy())
#print("Sample batch:", batch)


  0%|          | 0/40430 [00:00<?, ?it/s]

{'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[    1,  1167,   281,  2523,   271, 22642,   324,   874,   302,     2,
          1167,   281, 97251,   268,   324,   874,   302,     2,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,   

In [65]:
print(qqp['validation'][0])

{'text1': 'Why are African-Americans so beautiful?', 'text2': 'Why are hispanics so beautiful?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}


__Bonus:__ for bonus points, try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.